In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UNIVERSAL ENERGY SYSTEMS OPTIMIZATION FRAMEWORK v2.0 (FIXED)
# ═══════════════════════════════════════════════════════════════════════════════
# Fixes:
# 1. Separated Discovery Model (All Features) from Optimization Model (Inputs Only)
# 2. Ensures XGBoost feature names match exactly during optimization loops
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import warnings
import xgboost as xgb
import torch
import torch.nn as nn
import networkx as nx
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize, differential_evolution
from scipy.stats import qmc, norm

warnings.filterwarnings('ignore')

# GPU Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 UNIVERSAL ENERGY OPTIMIZATION FRAMEWORK")
print(f"   Device: {DEVICE}")
print("═" * 80)

# ═══════════════════════════════════════════════════════════════════════════════
# PART 1: PHYSICS-BASED SIMULATION ENGINES
# ═══════════════════════════════════════════════════════════════════════════════

class SolarSystemSimulator:
    @staticmethod
    def simulate(panel_area, efficiency, tilt_angle, azimuth, irradiance, temp_ambient, n_sims=1000000):
        temp_cell = temp_ambient + 25
        temp_loss = 0.004 * (temp_cell - 25)
        optimal_tilt = 30
        angle_loss = 1 - 0.2 * abs(tilt_angle - optimal_tilt) / 90
        azimuth_loss = np.cos(np.radians(abs(azimuth - 180)))
        soiling = np.random.uniform(0.95, 0.98, n_sims)
        
        power = (panel_area * (efficiency/100) * irradiance * 
                (1 - temp_loss) * angle_loss * azimuth_loss * soiling) / 1000
        
        return {
            'power_output_kw': power.mean(),
            'power_std': power.std(),
            'capacity_factor': power.mean() / (panel_area * (efficiency/100) * 1.0),
            'efficiency_effective': power.mean() / (panel_area * irradiance / 1000) * 100
        }

class NuclearReactorSimulator:
    @staticmethod
    def simulate(fuel_enrichment, coolant_temp, coolant_pressure, control_rod_position, core_size, n_sims=1000000):
        k_eff = 0.9 + (fuel_enrichment - 3) * 0.05 + (control_rod_position - 50) * 0.001
        k_eff = np.clip(k_eff, 0.95, 1.05)
        coolant_density = 700 + (17 - coolant_pressure) * 10
        moderation = coolant_density / 750
        temp_feedback = 1 - (coolant_temp - 300) * 0.001
        
        power_density = 100 * k_eff * moderation * temp_feedback
        thermal_power = core_size * power_density * np.random.normal(1.0, 0.02, n_sims)
        
        return {
            'thermal_power_mw': thermal_power.mean(),
            'electrical_power_mw': thermal_power.mean() * 0.33,
            'k_effective': k_eff,
            'capacity_factor': np.clip(k_eff, 0, 1)
        }

class CombustionEngineSimulator:
    @staticmethod
    def simulate(displacement, compression_ratio, fuel_injection_timing, air_fuel_ratio, rpm, n_sims=1000000):
        gamma = 1.4
        thermal_eff = 1 - (1 / compression_ratio**(gamma - 1))
        vol_eff = np.clip(0.85 - (rpm - 3500)**2 / 50000000, 0.65, 0.95)
        combustion_eff = np.clip(0.95 - abs(air_fuel_ratio - 14.7) * 0.02, 0.75, 0.98)
        timing_eff = 1 - abs(fuel_injection_timing - 15) * 0.01
        
        air_mass = displacement * vol_eff * 1.2 / 1000
        energy = (air_mass / air_fuel_ratio) * 43 * combustion_eff * timing_eff
        power_kw = energy * (rpm / 60) / 2 * 1000 * np.random.normal(1.0, 0.05, n_sims)
        
        return {
            'power_kw': power_kw.mean(),
            'torque_nm': (power_kw.mean() * 1000) / (rpm * 2 * np.pi / 60),
            'thermal_efficiency_%': thermal_eff * combustion_eff * timing_eff * 100
        }

class ElectricMotorSimulator:
    @staticmethod
    def simulate(rotor_diameter, stator_length, winding_turns, magnet_strength, voltage, current, rpm, n_sims=1000000):
        k_t = winding_turns * magnet_strength * (rotor_diameter/100)**2 * (stator_length/100)
        torque = k_t * current
        omega = rpm * 2 * np.pi / 60
        mech_power = torque * omega / 1000
        
        # Losses
        copper_loss = current**2 * (0.01 * winding_turns / (rotor_diameter * stator_length))
        iron_loss = 0.001 * (rpm/1000)**2 * (rotor_diameter * stator_length)
        
        eff = (mech_power / (mech_power + (copper_loss + iron_loss)/1000)) * 100
        power_out = mech_power * np.random.normal(1.0, 0.02, n_sims)
        
        return {
            'power_kw': power_out.mean(),
            'torque_nm': torque,
            'efficiency_%': np.clip(eff, 0, 99.9)
        }

class HybridPowertrainSimulator:
    @staticmethod
    def simulate(ice_displacement, motor_power_kw, battery_capacity_kwh, power_split_ratio, vehicle_speed_kmh, n_sims=1000000):
        # ICE Component
        ice_res = CombustionEngineSimulator.simulate(ice_displacement, 13.0, 15, 14.7, 1000 + vehicle_speed_kmh*30, 1)
        ice_p = ice_res['power_kw'] * power_split_ratio
        
        # Electric Component
        motor_res = ElectricMotorSimulator.simulate(20, 25, 100, 1.2, 400, motor_power_kw*2.5, vehicle_speed_kmh*100, 1)
        motor_p = motor_res['power_kw'] * (1 - power_split_ratio)
        
        total = (ice_p + motor_p) * np.random.normal(1.0, 0.03, n_sims)
        
        return {
            'total_power_kw': total.mean(),
            'system_efficiency_%': power_split_ratio * ice_res['thermal_efficiency_%'] + (1-power_split_ratio) * motor_res['efficiency_%'],
            'fuel_consumption_l_h': ice_p * 0.3
        }

# ═══════════════════════════════════════════════════════════════════════════════
# PART 2: OPTIMIZATION ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

class UnifiedOptimizer:
    def __init__(self, objective_fn, bounds):
        self.objective_fn = objective_fn
        self.bounds = bounds
        self.param_names = list(bounds.keys())
        self.history = {'params': [], 'values': []}

    def _dict_to_array(self, p):
        return np.array([p[k] for k in self.param_names])

    def _array_to_dict(self, a):
        return dict(zip(self.param_names, a))

    def bayesian_optimize(self, n_init=10, n_iter=20):
        # Initial Sampling
        sampler = qmc.LatinHypercube(d=len(self.param_names))
        sample = sampler.random(n=n_init)
        X, y = [], []
        
        for s in sample:
            p = {k: v[0] + s[i] * (v[1] - v[0]) for i, (k, v) in enumerate(self.bounds.items())}
            val = self.objective_fn(**p)
            X.append(self._dict_to_array(p))
            y.append(val)
            
        best_val = max(y)
        
        # Optimization Loop
        for i in range(n_iter):
            gp = GaussianProcessRegressor(kernel=C(1.0) * Matern(length_scale=1.0), n_restarts_optimizer=5)
            gp.fit(np.array(X), np.array(y))
            
            # Acquisition Function (EI)
            candidates = sampler.random(n=500)
            best_acq, best_cand = -np.inf, None
            
            for c in candidates:
                p_cand = {k: v[0] + c[j] * (v[1] - v[0]) for j, (k, v) in enumerate(self.bounds.items())}
                p_arr = self._dict_to_array(p_cand).reshape(1, -1)
                mu, sigma = gp.predict(p_arr, return_std=True)
                
                with np.errstate(divide='warn'):
                    imp = mu - best_val
                    Z = imp / (sigma + 1e-9)
                    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
                
                if ei > best_acq:
                    best_acq = ei
                    best_cand = p_cand
            
            val = self.objective_fn(**best_cand)
            X.append(self._dict_to_array(best_cand))
            y.append(val)
            best_val = max(best_val, val)
            
        best_idx = np.argmax(y)
        return self._array_to_dict(X[best_idx]), y[best_idx]

# ═══════════════════════════════════════════════════════════════════════════════
# PART 3: MASTER PIPELINE (CORRECTED)
# ═══════════════════════════════════════════════════════════════════════════════

def optimize_energy_system(system_name, simulator, bounds, n_samples=20000):
    print("\n" + "═" * 80)
    print(f"OPTIMIZING: {system_name.upper()}")
    print("═" * 80)
    
    # [STEP 1] DATA GENERATION
    print(f"[STEP 1] Generating {n_samples} simulations...")
    data = []
    param_names = list(bounds.keys())
    
    for _ in range(n_samples):
        p = {k: np.random.uniform(v[0], v[1]) for k, v in bounds.items()}
        res = simulator(**p, n_sims=1)
        data.append({**p, **res})
        
    df = pd.DataFrame(data)
    target_col = [c for c in df.columns if c not in param_names][0]
    print(f"   Target: {target_col}")
    
    # [STEP 2] DISCOVERY (Relationships)
    print("[STEP 2] Discovery & Feature Importance...")
    X_disc = df.drop(columns=[target_col]) # Discovery sees everything except target
    y_disc = df[target_col]
    
    disc_model = xgb.XGBRegressor(n_estimators=100, max_depth=4)
    disc_model.fit(X_disc, y_disc)
    
    imp = pd.Series(disc_model.feature_importances_, index=X_disc.columns).sort_values(ascending=False)
    print("   Top Drivers:")
    print(imp.head(3).to_string())
    
    # [STEP 3] VALIDATION
    print("[STEP 3] Data Validation...")
    iso = IsolationForest(contamination=0.05).fit(df.select_dtypes(include=np.number))
    anom = (iso.predict(df.select_dtypes(include=np.number)) == -1).mean()
    print(f"   Anomaly Rate: {anom:.1%}")
    
    # [STEP 4] OPTIMIZATION (THE FIX)
    print("[STEP 4] Optimization...")
    
    # CRITICAL FIX: Train a SURROGATE model that uses ONLY INPUTS (param_names)
    # This ensures the optimizer can query it using only the variables it controls.
    X_opt = df[param_names]
    y_opt = df[target_col]
    
    surrogate_model = xgb.XGBRegressor(n_estimators=200, max_depth=6)
    surrogate_model.fit(X_opt, y_opt)
    print(f"   Surrogate Model R²: {surrogate_model.score(X_opt, y_opt):.4f}")
    
    def objective_wrapper(**params):
        # Convert dict to DataFrame with correct column order for XGBoost
        row = pd.DataFrame([params], columns=param_names)
        return surrogate_model.predict(row)[0]
    
    optimizer = UnifiedOptimizer(objective_wrapper, bounds)
    best_params, best_val = optimizer.bayesian_optimize()
    
    print(f"   🏆 Optimized {target_col}: {best_val:.4f}")
    
    # [STEP 5] ATTRIBUTION
    print("[STEP 5] Attribution (Integrated Gradients)...")
    try:
        # Simple attribution using surrogate weights/importance for speed
        attr = pd.Series(surrogate_model.feature_importances_, index=param_names).sort_values(ascending=False)
        print("   Key Inputs:")
        print(attr.head(3).to_string())
    except:
        pass
        
    return {
        'system': system_name, 
        'best_params': best_params, 
        'performance': best_val,
        'drivers': imp.head(5).to_dict()
    }

# ═══════════════════════════════════════════════════════════════════════════════
# EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    
    # 1. SOLAR
    solar_res = optimize_energy_system(
        "Solar PV", SolarSystemSimulator.simulate,
        {'panel_area': (10, 100), 'efficiency': (15, 25), 'tilt_angle': (0, 60), 
         'azimuth': (135, 225), 'irradiance': (400, 1000), 'temp_ambient': (15, 35)}
    )
    
    # 2. NUCLEAR
    nuc_res = optimize_energy_system(
        "Nuclear PWR", NuclearReactorSimulator.simulate,
        {'fuel_enrichment': (3.5, 4.5), 'coolant_temp': (290, 320), 
         'coolant_pressure': (15.5, 16.5), 'control_rod_position': (60, 95), 'core_size': (150, 250)}
    )
    
    # 3. ICE
    ice_res = optimize_energy_system(
        "Combustion Engine", CombustionEngineSimulator.simulate,
        {'displacement': (1.5, 5.0), 'compression_ratio': (9, 12), 
         'fuel_injection_timing': (10, 20), 'air_fuel_ratio': (13.5, 15.5), 'rpm': (2000, 6000)}
    )
    
    # 4. ELECTRIC MOTOR
    motor_res = optimize_energy_system(
        "Electric Motor", ElectricMotorSimulator.simulate,
        {'rotor_diameter': (15, 40), 'stator_length': (15, 35), 'winding_turns': (80, 150),
         'magnet_strength': (1.0, 1.4), 'voltage': (300, 600), 'current': (200, 500), 'rpm': (3000, 15000)}
    )
    
    # 5. HYBRID
    hybrid_res = optimize_energy_system(
        "Hybrid Powertrain", HybridPowertrainSimulator.simulate,
        {'ice_displacement': (1.5, 2.5), 'motor_power_kw': (40, 80), 'battery_capacity_kwh': (1.5, 5),
         'power_split_ratio': (0.3, 0.7), 'vehicle_speed_kmh': (40, 100)}
    )

    print("\n" + "═" * 80)
    print("🎉 FINAL RESULTS SUMMARY")
    print("═" * 80)
    results = [solar_res, nuc_res, ice_res, motor_res, hybrid_res]
    df_res = pd.DataFrame([{
        'System': r['system'], 
        'Optimized Perf': r['performance'],
        'Top Driver': list(r['drivers'].keys())[0]
    } for r in results])
    print(df_res.to_string(index=False))

In [ ]:
z# ═══════════════════════════════════════════════════════════════════════════════
# PART 9: AUTOMATED REVERSE ENGINEERING & COMPONENT MATCHING
# ═══════════════════════════════════════════════════════════════════════════════
# This module takes the mathematical optimization results and:
# 1. Matches them to real-world commercial components from a lookup database.
# 2. Extracts and displays the governing physics equations used for that specific system.
# ═══════════════════════════════════════════════════════════════════════════════

def reverse_engineer_system(system_name, optimal_params, optimal_performance):
    print("\n" + "═" * 80)
    print(f"🔧 REVERSE ENGINEERING: {system_name.upper()}")
    print("═" * 80)
    
    # -----------------------------------------------------------------------
    # 1. COMPONENT DATABASE (Mini Knowledge Base)
    # -----------------------------------------------------------------------
    component_db = {
        "Solar PV": [
            {"name": "Residential Panel (Standard)", "specs": {"panel_area": 1.6, "efficiency": 18, "type": "Polycrystalline"}},
            {"name": "Commercial High-Efficiency", "specs": {"panel_area": 2.0, "efficiency": 22, "type": "Monocrystalline"}},
            {"name": "Utility Scale Array (Small)", "specs": {"panel_area": 50.0, "efficiency": 20, "type": "Thin Film/Mono Hybrid"}},
            {"name": "Utility Scale Array (Large)", "specs": {"panel_area": 100.0, "efficiency": 24, "type": "Bifacial Monocrystalline"}}
        ],
        "Nuclear PWR": [
            {"name": "Small Modular Reactor (SMR)", "specs": {"thermal_power_mw": 200, "fuel_enrichment": 4.8, "core_size": 15}},
            {"name": "Standard Gen II Reactor", "specs": {"thermal_power_mw": 3000, "fuel_enrichment": 3.5, "core_size": 100}},
            {"name": "Gen III+ Massive Reactor", "specs": {"thermal_power_mw": 4500, "fuel_enrichment": 4.5, "core_size": 150}},
            {"name": "Theoretical Gen IV HALEU Core", "specs": {"thermal_power_mw": 20000, "fuel_enrichment": 4.95, "core_size": 250}}
        ],
        "Combustion Engine": [
            {"name": "Economy I4 (1.5L)", "specs": {"displacement": 1.5, "rpm": 5500, "cylinders": 4}},
            {"name": "Sport V6 (3.5L)", "specs": {"displacement": 3.5, "rpm": 6500, "cylinders": 6}},
            {"name": "Muscle V8 (5.0L)", "specs": {"displacement": 5.0, "rpm": 7000, "cylinders": 8}},
            {"name": "Racing V12 (6.5L)", "specs": {"displacement": 6.5, "rpm": 8500, "cylinders": 12}}
        ],
        "Electric Motor": [
            {"name": "Small EV Motor", "specs": {"power_kw": 100, "voltage": 350, "torque_nm": 250}},
            {"name": "Performance EV Dual-Motor", "specs": {"power_kw": 400, "voltage": 400, "torque_nm": 600}},
            {"name": "Industrial Locomotive Motor", "specs": {"power_kw": 1500, "voltage": 1000, "torque_nm": 5000}},
            {"name": "Marine Propulsion Pod", "specs": {"power_kw": 4000, "voltage": 800, "torque_nm": 20000}}
        ],
        "Hybrid Powertrain": [
            {"name": "City Hybrid (Prius-style)", "specs": {"ice_displacement": 1.8, "motor_power_kw": 50, "battery_kwh": 1.0}},
            {"name": "PHEV Crossover (RAV4 Prime)", "specs": {"ice_displacement": 2.5, "motor_power_kw": 130, "battery_kwh": 18.0}},
            {"name": "Hypercar Hybrid (P1/918)", "specs": {"ice_displacement": 4.6, "motor_power_kw": 200, "battery_kwh": 5.0}}
        ]
    }
    
    # -----------------------------------------------------------------------
    # 2. FIND NEAREST MATCH
    # -----------------------------------------------------------------------
    candidates = component_db.get(system_name, [])
    best_match = None
    min_diff = float('inf')
    
    print("🔍 MATCHING TO REAL-WORLD COMPONENTS:")
    
    for cand in candidates:
        diff_score = 0
        comparisons = []
        
        # Compare based on available specs
        for key, val in cand['specs'].items():
            if key in optimal_params:
                # Parameter match (e.g., displacement)
                sim_val = optimal_params[key]
                diff = abs(sim_val - val) / val
                diff_score += diff
                comparisons.append(f"{key}: {sim_val:.1f} vs {val:.1f}")
            elif key == "power_kw" or key == "thermal_power_mw":
                # Output match
                sim_val = optimal_performance
                diff = abs(sim_val - val) / val
                diff_score += diff
                comparisons.append(f"Output: {sim_val:.0f} vs {val:.0f}")
                
        if diff_score < min_diff:
            min_diff = diff_score
            best_match = cand
            best_comparison = comparisons

    if best_match:
        print(f"   ✅ Closest Match: {best_match['name']}")
        print(f"      Confidence: {max(0, 100 - min_diff*20):.1f}%")
        print(f"      Basis: {', '.join(best_comparison)}")
    else:
        print("   ❌ No close commercial match found (Custom/Theoretical Design)")

    # -----------------------------------------------------------------------
    # 3. REVEAL PHYSICS EQUATIONS
    # -----------------------------------------------------------------------
    print("\n📐 GOVERNING PHYSICS EQUATIONS (REVERSE ENGINEERED):")
    
    if system_name == "Solar PV":
        print("   1. Power Output:")
        print("      P = Area * (Eff/100) * Irradiance * (1 - L_temp) * L_angle")
        print("   2. Temperature Loss (Linear Model):")
        print("      L_temp = 0.004 * (T_ambient + 25 - 25)")
        print("   3. Angle of Incidence Loss:")
        print("      L_angle = 1 - 0.2 * |Tilt - 30| / 90")
        
    elif system_name == "Nuclear PWR":
        print("   1. Neutron Multiplication (k_eff):")
        print("      k = 0.9 + (Enrichment - 3)*0.05 + (Rod_Pos - 50)*0.001")
        print("   2. Coolant Density (Moderation):")
        print("      rho = 700 + (17 - Pressure_MPa) * 10")
        print("   3. Thermal Power:")
        print("      P_th = Core_Vol * (100 * k * (rho/750))")
        
    elif system_name == "Combustion Engine":
        print("   1. Otto Cycle Efficiency:")
        print("      eta_th = 1 - (1 / Compression_Ratio^(0.4))")
        print("   2. Volumetric Efficiency (Air Flow):")
        print("      eta_vol = 0.85 - (RPM - 3500)^2 / 5e7")
        print("   3. Power (4-Stroke):")
        print("      P = (Fuel_Energy * eta_th * eta_vol) * (RPM / 120)")
        
    elif system_name == "Electric Motor":
        print("   1. Torque Constant (k_t):")
        print("      k_t = Turns * B_field * Radius^2 * Length")
        print("   2. Back EMF:")
        print("      V_emf = k_t * (RPM * 2pi / 60)")
        print("   3. Copper Loss (I^2R):")
        print("      P_loss = I^2 * (Resistance_const * Turns / Area)")
        
    elif system_name == "Hybrid Powertrain":
        print("   1. Power Split Logic:")
        print("      P_total = P_ice(RPM_ice) * Ratio + P_motor(RPM_motor) * (1-Ratio)")
        print("   2. Speed Coupling:")
        print("      RPM_ice = 1000 + Speed_kmh * 30")
        print("      RPM_motor = Speed_kmh * 100")
        
    print("\n" + "═" * 80)

# -----------------------------------------------------------------------
# EXECUTE REVERSE ENGINEERING FOR ALL SYSTEMS
# -----------------------------------------------------------------------
# This uses the results list from the previous code block
if 'results' in locals():
    for res in results:
        reverse_engineer_system(
            res['system'], 
            res['best_params'], 
            res['performance']
        )
else:
    print("⚠️ Run the optimization code block first to generate results!")
